## Executive summary (read this first)

這份 Notebook 只讀取已完成的實驗，不修改參數、不呼叫 House、不送出競賽提交。
請用 run-windows.ps1 -Phase notebook 開啟，它會將本檔複製到結果目錄。
分數是相對於本地300觀測漂移基準的診斷損失，越低越好；不是官方排行榜分數。
資料先前已被探索過，後期資料是這次鎖定的測試，不能宣稱完全未看過的正式 OOS。


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
RUN_DIR = Path.cwd()  # 若手動開啟，改成你的結果目錄
assert (RUN_DIR / "plan.json").exists(), "請將 RUN_DIR 改成實驗結果目錄"
plan = json.loads((RUN_DIR / "plan.json").read_text(encoding="utf-8"))
selected = json.loads((RUN_DIR / "selected.json").read_text(encoding="utf-8"))
print("Selected:", selected["config"])
print("Draws:", plan["draws"], "Seeds:", plan["seeds"], "Rankable:", plan["rankable"])
print("Selection dates:", plan["selection_origins"])
print("Later test dates:", plan["holdout_origins"])


### 選參數階段
每個資產組合先平均，再給每個組合相同權重，避免重複卡片造成虛假的優勢。
此表用來選候選，不能再被稱為獨立驗證。


In [ ]:
summary = pd.read_csv(RUN_DIR / "selection_summary.csv")
display(summary)
ax = summary.set_index("config")["local_relative_loss"].plot.barh(figsize=(9, 7))
ax.axvline(1, color="black", linestyle="--", label="Local baseline")
ax.invert_yaxis(); ax.set_xlabel("Local relative loss (lower is better)")
ax.legend(); plt.tight_layout(); plt.show()


### 分類與種子穩定性
每種子的平均不代表統計獨立樣本。Family 是來源卡的分類；歷史模擬日期不一定是該類的政策衝擊。


In [ ]:
diagnostics = pd.read_csv(RUN_DIR / "selection_by_family_seed.csv")
display(diagnostics[diagnostics.config.isin([selected["config"], "w300_d1_s1", "calibrated_policy"])])
coverage = json.loads((RUN_DIR / "coverage.json").read_text(encoding="utf-8"))
print("Deduplicated source grids:", coverage["selected_source_units"])
print("Inventory exclusions:", len(coverage["inventory_skipped"]))
print("Selection case exclusions:", len(coverage["selection_excluded"]))
display(pd.DataFrame(coverage["selection_excluded"]).groupby("reason").size().rename("count"))


### 鎖定後的測試
需先執行 run-windows.ps1 -Phase holdout。只看已選候選與基準；不替其他19組計算後期分數。
Bootstrap 區間以資產組合重抽，但組合可能共享資產，所以只能視為描述性不確定性。
若候選在這段退步，記錄失敗，不依這份表重新挑第二名。


In [ ]:
if (RUN_DIR / "holdout_summary.csv").exists():
    display(pd.read_csv(RUN_DIR / "holdout_summary.csv"))
    print(json.dumps(json.loads((RUN_DIR / "paired_interval.json").read_text()), indent=2))
    display(pd.read_csv(RUN_DIR / "holdout_by_family_seed.csv"))
else:
    print("Holdout 尚未執行；選定候選後才執行。")


### 版本核對
以 audit_submission.py 讀取實際上傳的 ZIP。將 Submission ID、映像 digest 與分數一起記錄。
不要把 Team Key、team-claim.json、ZIP 或這份已執行 Notebook 推上公開 GitHub。
